# 04 — Ablations (w/o nutrition inject, w/o flavor compound)

Two ablation rows for Table 2 of the report:
- **v1 MVP**: keeps `g` + `L_health` but no architectural nutrient injection (Ablation 1, w/o nutrition edge).
- **v3 with `--ablation_no_compound`**: structural hub kept but I-F / I-D edges dropped (Ablation 3, w/o flavor compound).

Ablation 2 (w/o L_health) is automatically covered by the lambda=0 row of the v3 sweep -- nothing extra to run here.

**Runtime**: Colab free T4 GPU, ~1 hr.

**Steps**:
1. Runtime > Change runtime type > T4 GPU
2. Set `PROJECT_ROOT` (cell 4) if your Drive layout differs
3. Runtime > Run all

## 1. GPU + dependencies

In [ ]:
!nvidia-smi

In [ ]:
# torch_geometric needs to be matched with the installed torch wheel.
# Colab's default torch is recent enough that the generic install works.
!pip install -q torch_geometric pandas numpy matplotlib

## 2. Mount Drive (code + data both live in Drive)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
PROJECT_ROOT = '/content/drive/MyDrive/CS471_project'
CODE_DIR     = f'{PROJECT_ROOT}/code'
DATA_DIR     = f'{PROJECT_ROOT}/data'
OUTPUT_DIR   = f'{PROJECT_ROOT}/outputs'
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.chdir(CODE_DIR)
print(f'CWD        = {os.getcwd()}')
print(f'DATA_DIR   = {DATA_DIR}')
print(f'OUTPUT_DIR = {OUTPUT_DIR}')

## 3. Smoke tests

In [ ]:
# 1-epoch smoke test (~30 sec on T4). Skip by changing False below.
RUN_SMOKE = True
if RUN_SMOKE:
    !python src/train_v1.py --mode mvp --max_epochs 1 --patience 1 --no_resume \
      --data_dir {DATA_DIR} --output_dir {OUTPUT_DIR}/smoke_v1_mvp
    print('\n[smoke] OK')

In [ ]:
# 1-epoch smoke test (~30 sec on T4). Skip by changing False below.
RUN_SMOKE = True
if RUN_SMOKE:
    !python src/train_v3.py --max_epochs 1 --patience 1 --no_resume \
      --data_dir {DATA_DIR} --output_dir {OUTPUT_DIR}/smoke_v3
    print('\n[smoke] OK')

## 4. Train v1 MVP (Ablation 1)

In [ ]:
# All g overrides included so the case study cell in 06 works.
!python src/train_v1.py --mode mvp --tau_percentile 0 \
  --test_g_overrides auto 1_0 0_1 1_1 \
  --data_dir {DATA_DIR} --output_dir {OUTPUT_DIR}/v1mvp

## 5. Train v3 with no compound edges (Ablation 3)

In [ ]:
!python src/train_v3.py --tau_percentile 0 --ablation_no_compound \
  --data_dir {DATA_DIR} --output_dir {OUTPUT_DIR}/v3_no_compound

## 6. Quick check

In [ ]:
import json
for label, path in [
    ('v1 MVP', f'{OUTPUT_DIR}/v1mvp/test_predictions_mvp_auto.json'),
    ('v3 no_compound', f'{OUTPUT_DIR}/v3_no_compound/test_predictions_v3_auto.json'),
]:
    with open(path) as f:
        m = json.load(f)['metrics']
    print(f'{label:<20s} MRR={m["MRR"]:.2f}  Hit@10={m["Hit@10"]:.2f}')